In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 Chrome/124.0"}

def get_zoo_links():
    r = requests.get("https://www.zoofrance.com/carte/", headers=HEADERS)
    soup = BeautifulSoup(r.text, "html.parser")
    zoos = {}
    for li in soup.select("ul li a[href*='zoofrance.com']"):
        name = li.get_text(strip=True)
        href = li["href"]
        if name and "/carte" not in href:
            zoos[name] = href
    return zoos

def extract_info(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(r.text, "html.parser")

    adresse = None
    lat, lon = None, None

    # Chercher les balises <strong> qui contiennent "Adresse"
    for strong in soup.find_all("strong"):
        if "Adresse" in strong.get_text():
            # Le texte de l'adresse est dans le noeud suivant (NavigableString)
            next_node = strong.next_sibling
            if next_node:
                adresse = str(next_node).strip().lstrip(": ").strip()
            break

    # Coordonnées GPS
    gps_match = re.search(r"latitude\s*([\d.]+)\s*\|\s*longitude\s*([\d.]+)", soup.get_text())
    if gps_match:
        lat = float(gps_match.group(1))
        lon = float(gps_match.group(2))

    return {"adresse": adresse, "latitude": lat, "longitude": lon}

zoos = get_zoo_links()
records = []
for name, url in zoos.items():
    print(f"→ {name}")
    info = extract_info(url)
    records.append({"nom": name, "url": url, **info})
    time.sleep(0.8)

df = pd.DataFrame(records)
df.to_csv("zoos_france2.csv", index=False, encoding="utf-8-sig")
print(df[["nom", "adresse", "latitude", "longitude"]])

→ Accueil
→ Classement
→ Promo Billets
→ Beauval
→ Tarifs 2026
→ Séjours Hôtels
→ Prévisions fréquentation
→ Paris
→ Le PAL
→ Thoiry
→ Amnéville
→ La Flèche
→ Nausicaa
→ Sigean
→ Peaugres
→ La Palmyre
→ Lumigny
→ Barben
→ Doué la Fontaine
→ Aquarium de Paris
→ Aquarium de Saint-Malo
→ Aquarium La Rochelle
→ Bioparc de Doué la Fontaine
→ Biotropica
→ La Ferme aux Crocodiles
→ La Vallée des Singes
→ Les Terres de Nataé
→ Lumigny Safari Réserve
→ Océanopolis
→ Parc animalier de Sainte-Croix
→ Parc animalier des Pyrénées
→ Parc de Branféré
→ Parc de Merlet
→ Parc des Oiseaux
→ Parc Zoologique de Paris
→ Parc Zoo du Reynou
→ Planète Sauvage
→ Planet Exotica
→ Réserve de Sigean
→ Safari de Peaugres
→ Seaquarium
→ Touroparc
→ Zoo African Safari
→ Zoo d’Amiens
→ Zoo d’Amnéville
→ Zoo de Cerza
→ Zoo de Champrépus
→ Zoo de Jurques
→ Zoo de la Barben
→ Zoo de la Boissière du Doré
→ Zoo de la Flèche
→ Zoo de La Palmyre
→ Zoo de Lille
→ Zoo de Lyon
→ Zoo de Montpellier
→ Zoo de Mulhouse
→ Zoo des S

In [12]:
df_zoo = pd.read_csv("C:/Users/jadeb/projet 3/MoveinLoc/zoos_france2.csv")

In [14]:
df_zoo.head(15)

,nom,url,adresse,latitude,longitude
0,Accueil,https://www.zoofrance.com/,NaN,NaN,NaN
1,Classement,https://www.zoofrance.com/classement-zoos-france/,NaN,NaN,NaN
2,Promo Billets,https://www.zoofrance.com/zoo-beauval/promo-bi...,NaN,NaN,NaN
3,Beauval,https://www.zoofrance.com/zoo-beauval/,Avenue du Blanc 41110 Saint-Aignan-sur-Cher,47.247579,1.353391
4,Tarifs 2026,https://www.zoofrance.com/zoo-beauval/tarifs-b...,NaN,NaN,NaN
5,Séjours Hôtels,https://www.zoofrance.com/zoo-beauval/hotels/,Route de Beauval 41110 Saint-Aignan-sur-Cher,NaN,NaN
6,Prévisions fréquentation,https://www.zoofrance.com/zoo-beauval/previsio...,NaN,NaN,NaN
7,Paris,https://www.zoofrance.com/zoo-paris-vincennes/,route de la Ceinture du Lac Daumesnil 75012 Paris,48.833750,2.412730
8,Le PAL,https://www.zoofrance.com/zoo-le-pal/,Saint-Pourçain-sur-Besbre CS 60001 03290 Dompi...,NaN,NaN
9,Thoiry,https://www.zoofrance.com/zoo-thoiry/,rue du Pavillon de Montreuil 78770 Thoiry,48.863400,1.797880


In [15]:
df_zoo.info()

<class 'pandas.DataFrame'>
RangeIndex: 61 entries, 0 to 60
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   nom        61 non-null     str    
 1   url        61 non-null     str    
 2   adresse    56 non-null     str    
 3   latitude   28 non-null     float64
 4   longitude  28 non-null     float64
dtypes: float64(2), str(3)
memory usage: 8.2 KB


In [16]:
df_zoo = df.dropna(subset="adresse")

In [17]:
df_zoo

,nom,url,adresse,latitude,longitude
3,Beauval,https://www.zoofrance.com/zoo-beauval/,Avenue du Blanc 41110 Saint-Aignan-sur-Cher,47.247579,1.353391
5,Séjours Hôtels,https://www.zoofrance.com/zoo-beauval/hotels/,Route de Beauval 41110 Saint-Aignan-sur-Cher,NaN,NaN
7,Paris,https://www.zoofrance.com/zoo-paris-vincennes/,route de la Ceinture du Lac Daumesnil 75012 Paris,48.833750,2.412730
8,Le PAL,https://www.zoofrance.com/zoo-le-pal/,Saint-Pourçain-sur-Besbre CS 60001 03290 Dompi...,NaN,NaN
9,Thoiry,https://www.zoofrance.com/zoo-thoiry/,rue du Pavillon de Montreuil 78770 Thoiry,48.863400,1.797880
10,Amnéville,https://www.zoofrance.com/zoo-amneville/,1 rue du Tigre 57360 Amnéville,49.245553,6.137827
11,La Flèche,https://www.zoofrance.com/zoo-la-fleche/,Le Tertre Rouge 72200 La Flèche,NaN,NaN
12,Nausicaa,https://www.zoofrance.com/nausicaa-boulogne-su...,Boulevard Sainte Beuve 62203 Boulogne-sur-Mer,50.730934,1.595746
13,Sigean,https://www.zoofrance.com/reserve-africaine-si...,19 Chemin Hameau du lac 11130 Sigean,43.062710,2.949801
14,Peaugres,https://www.zoofrance.com/safari-peaugres/,Safari de Peaugres 07340 Peaugres,45.269590,4.713520


In [ ]:
df_zoo.to_csv("zoo_france.csv")

: 